In [ ]:
from google.colab import auth, drive
import os

auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)

# Points directly to the exact dataset folder you mentioned
DRIVE_IMAGE_FOLDER = 'fire_detection/DataSet'
DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_IMAGE_FOLDER}'

assert os.path.isdir(DRIVE_DIR), (
    f'Folder not found: {DRIVE_DIR}\n'
    f'Make sure "{DRIVE_IMAGE_FOLDER}" exists inside My Drive on Google Drive.'
)

IMAGE_DIR = DRIVE_DIR
print(f'Reading images directly from {IMAGE_DIR}')

total = sum(1 for _, _, fs in os.walk(IMAGE_DIR) for f in fs)
print(f'Found {total} file(s) in total (Dangerous_fire, Looks_Like_Fire, Safe_fire)')


Mounted at /content/drive
Reading images directly from /content/drive/MyDrive/fire_detection/DataSet
Found 190 file(s) in total (Dangerous_fire, Looks_Like_Fire, Safe_fire)


In [ ]:
!pip install -q "transformers==4.45.2" accelerate einops timm Pillow opencv-python-headless numpy torch torchvision
print('Done.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 121.1 MB/s eta 0:00:00
Done.


In [ ]:
import pathlib, sys

CACHE_ROOT = pathlib.Path.home() / '.cache/huggingface/modules/transformers_modules/OpenGVLab'

BAD_LINE  = 'dpr = [x.item() for x in torch.linspace(0, config.drop_path_rate, config.num_hidden_layers)]'
GOOD_LINE = ('_n   = int(config.num_hidden_layers)\n'
             '        _end = config.drop_path_rate\n'
             '        _end = _end if isinstance(_end, float) else 0.1\n'
             '        dpr  = [_end * i / max(_n - 1, 1) for i in range(_n)]')

patched = 0
for vit_file in CACHE_ROOT.rglob('modeling_intern_vit.py'):
    code = vit_file.read_text(encoding='utf-8')
    if BAD_LINE in code:
        vit_file.write_text(code.replace(BAD_LINE, GOOD_LINE), encoding='utf-8')
        print(f'Patched file: {vit_file}')
        patched += 1
    elif '_end = config.drop_path_rate' in code:
        print(f'Already patched: {vit_file}')
        patched += 1

evicted = [k for k in list(sys.modules) if 'internvl' in k.lower() or 'intern_vit' in k.lower()]
for k in evicted:
    del sys.modules[k]
print('Ready — run Cell 4 now.')


Ready — run Cell 4 now.


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_ID = 'OpenGVLab/InternVL2_5-4B'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE    = torch.bfloat16 if DEVICE == 'cuda' else torch.float32

print(f'Loading {MODEL_ID} on {DEVICE.upper()} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
model     = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=False,
).eval().to(DEVICE)

print(f'Model ready  |  device: {DEVICE.upper()}  |  dtype: {DTYPE}')


Loading OpenGVLab/InternVL2_5-4B on CUDA ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.43G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Model ready  |  device: CUDA  |  dtype: torch.bfloat16


In [ ]:
MAX_NEW_TOKENS   = 40
IMAGE_SIZE       = 448
MAX_TILES        = 6

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.wmv', '.m4v'}

# The highly-constrained 3-tier classification prompt under 10 words
PROMPT = (
    "Write one short, concise sentence (strictly under 10 words) describing the image. "
    "Your sentence MUST begin with exactly one of these three classification keywords: "
    "'dangerous_fire', 'safe_fire', or 'no_fire'. "
    "Use 'safe_fire' for small, controlled flames like candles or stoves. "
    "Use 'dangerous_fire' for uncontrolled fires, structural fires, or smoke hazards. "
    "Use 'no_fire' if there are no flames visible. "
    "The rest of your sentence must state the specific location (e.g., kitchen, forest) and primary related objects."
)

# Common English stop words. 'no' and 'not' are intentionally removed so 'no_fire' isn't ruined!
STOPWORDS = {
    'a','an','the','is','are','was','were','be','been','being',
    'in','on','at','to','of','for','with','by','from','into',
    'and','or','but','as','its','it','this','that',
    'there','their','they','has','have','had','do','does','did',
    'i','we','you','he','she','over','under','near','between',
    'through','around','against','during','without','within',
}


In [ ]:
import time
from pathlib import Path
import cv2
import numpy as np
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(size=IMAGE_SIZE):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB')),
        T.Resize((size, size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def dynamic_preprocess(image, min_num=1, max_num=MAX_TILES, image_size=IMAGE_SIZE):
    w, h    = image.size
    aspect  = w / h
    ratios  = sorted(
        {(i, j) for n in range(min_num, max_num + 1)
                for i in range(1, n + 1)
                for j in range(1, n + 1)
                if min_num <= i * j <= max_num},
        key=lambda x: x[0] * x[1]
    )
    best = min(ratios, key=lambda r: abs(aspect - r[0] / r[1]))
    tw, th  = image_size * best[0], image_size * best[1]
    resized = image.resize((tw, th))
    tiles   = [
        resized.crop((
            (idx % best[0]) * image_size, (idx // best[0]) * image_size,
            ((idx % best[0]) + 1) * image_size, ((idx // best[0]) + 1) * image_size,
        ))
        for idx in range(best[0] * best[1])
    ]
    tiles.append(image.resize((image_size, image_size)))
    return tiles

def load_pixel_values(image):
    transform = build_transform()
    tiles     = dynamic_preprocess(image.convert('RGB'))
    return torch.stack([transform(t) for t in tiles]).to(DTYPE).to(DEVICE)

def get_phrase(image):
    pixel_values = load_pixel_values(image)
    gen_cfg      = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    response     = model.chat(tokenizer, pixel_values, PROMPT, gen_cfg)
    return response.strip()

def analyse_phrase(phrase):
    words             = phrase.split()
    meaningful_tokens = [w.strip('.,;:!?\'"').lower()
                         for w in words
                         if w.strip('.,;:!?\'"').lower() not in STOPWORDS
                         and w.strip('.,;:!?\'"')]
    return {
        'word_count'        : len(words),
        'meaningful_tokens' : meaningful_tokens,
        'token_count'       : len(meaningful_tokens),
    }

print('Helper functions ready.')


Helper functions ready.


In [ ]:
import csv
root     = Path(IMAGE_DIR)
# Export to CSV inside your fire_detection folder
out_path = Path('/content/drive/MyDrive/fire_detection/internvl_tokens.csv')

all_files = sorted(
    p for p in root.rglob('*')
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS | VIDEO_EXTS
)

print(f'Processing {len(all_files)} file(s)...\n' + '─' * 60)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, 'w', encoding='utf-8', newline='') as out:
    writer = csv.writer(out)
    writer.writerow(['filename', 'category_folder', 'phrase', 'meaningful_tokens', 'word_count', 'meaningful_token_count'])

    for idx, path in enumerate(all_files, 1):
        rel = path.relative_to(root)
        # Pulls 'Safe_fire', 'Dangerous_fire', etc
        category_folder = rel.parent.name if rel.parent != Path('.') else 'root'
        ext = path.suffix.lower()
        print(f'[{idx}/{len(all_files)}] {rel}')

        try:
            phrase = get_phrase(Image.open(path))
            info   = analyse_phrase(phrase)

            tokens_str = ' '.join(info['meaningful_tokens'])

            writer.writerow([
                rel.name,
                category_folder,
                phrase,
                tokens_str,
                info['word_count'],
                info['token_count']
            ])
            out.flush()

            print(f'  phrase      : {phrase!r}')
            print(f'  meaningful  : {info["meaningful_tokens"]}')

        except Exception as e:
            print(f'  ERROR: {e}')
            writer.writerow([rel.name, category_folder, f'ERROR: {e}', '', '', ''])
            out.flush()

print(f'\nDone! Dataset tokens successfully saved to {out_path}')


Processing 200 file(s)...
────────────────────────────────────────────────────────────
[1/200] Looks_Like_Fire/PublicDataset00019.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


  phrase      : 'dangerous_fire in forest.'
  meaningful  : ['dangerous_fire', 'forest']
[2/200] Looks_Like_Fire/PublicDataset00085 - Copy - Copy.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


  phrase      : 'dangerous_fire in forest'
  meaningful  : ['dangerous_fire', 'forest']
[3/200] Looks_Like_Fire/PublicDataset00085 - Copy.jpg


...

  phrase      : 'dangerous_fire at night sky with fireworks.'
  meaningful  : ['dangerous_fire', 'night', 'sky', 'fireworks']
[199/200] Safe_fire/fireworks029.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


  phrase      : 'safe_fire: fireworks display in stadium.'
  meaningful  : ['safe_fire', 'fireworks', 'display', 'stadium']
[200/200] Safe_fire/fireworks030.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


  phrase      : 'dangerous_fire over water'
  meaningful  : ['dangerous_fire', 'water']

Done! Dataset tokens successfully saved to /content/drive/MyDrive/fire_detection/internvl_tokens.csv


In [ ]:
from google.colab import auth, drive
import os

auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)

# Points directly to the exact dataset folder you mentioned
DRIVE_IMAGE_FOLDER = 'fire_detection/DataSet'
DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_IMAGE_FOLDER}'

assert os.path.isdir(DRIVE_DIR), (
    f'Folder not found: {DRIVE_DIR}\n'
    f'Make sure "{DRIVE_IMAGE_FOLDER}" exists inside My Drive on Google Drive.'
)

IMAGE_DIR = DRIVE_DIR
print(f'Reading images directly from {IMAGE_DIR}')

total = sum(1 for _, _, fs in os.walk(IMAGE_DIR) for f in fs)
print(f'Found {total} file(s) in total (Dangerous_fire, Looks_Like_Fire, Safe_fire)')


Mounted at /content/drive
Reading images directly from /content/drive/MyDrive/fire_detection/DataSet
Found 200 file(s) in total (Dangerous_fire, Looks_Like_Fire, Safe_fire)


In [ ]:
from google.colab import auth, drive
import os

auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)

# Points directly to the exact dataset folder you mentioned
DRIVE_IMAGE_FOLDER = 'fire_detection/DataSet'
DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_IMAGE_FOLDER}'

assert os.path.isdir(DRIVE_DIR), (
    f'Folder not found: {DRIVE_DIR}\n'
    f'Make sure "{DRIVE_IMAGE_FOLDER}" exists inside My Drive on Google Drive.'
)

# Crucial fix: The directory itself is now locked down exclusively to Looks_Like_Fire
IMAGE_DIR = f'{DRIVE_DIR}/Looks_Like_Fire'
print(f'Reading images directly from {IMAGE_DIR}')

total = sum(1 for _, _, fs in os.walk(IMAGE_DIR) for f in fs)
print(f'Found {total} file(s) in Looks_Like_Fire')


Mounted at /content/drive
Reading images directly from /content/drive/MyDrive/fire_detection/DataSet/Looks_Like_Fire
Found 100 file(s) in Looks_Like_Fire


In [ ]:
import csv
root     = Path(IMAGE_DIR)
# Export to CSV inside your fire_detection folder
out_path = Path('/content/drive/MyDrive/fire_detection/internvl_Looks_Like_Fire_tokens.csv')

# Pulls strictly from root because root is now explicitly Looks_Like_Fire
all_files = sorted(
    p for p in root.rglob('*')
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS | VIDEO_EXTS
)

print(f'Processing {len(all_files)} file(s)...\n' + '─' * 60)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, 'w', encoding='utf-8', newline='') as out:
    writer = csv.writer(out)
    writer.writerow(['filename', 'category_folder', 'phrase', 'meaningful_tokens', 'word_count', 'meaningful_token_count'])

    for idx, path in enumerate(all_files, 1):
        rel = path.relative_to(root)

        # Ensures category_folder prints "Looks_Like_Fire" instead of "root"
        category_folder = rel.parent.name if rel.parent != Path('.') else 'Looks_Like_Fire'
        ext = path.suffix.lower()
        print(f'[{idx}/{len(all_files)}] {rel}')

        try:
            phrase = get_phrase(Image.open(path))
            info   = analyse_phrase(phrase)

            tokens_str = ' '.join(info['meaningful_tokens'])

            writer.writerow([
                rel.name,
                category_folder,
                phrase,
                tokens_str,
                info['word_count'],
                info['token_count']
            ])
            out.flush()

            print(f'  phrase      : {phrase!r}')
            print(f'  meaningful  : {info["meaningful_tokens"]}')

        except Exception as e:
            print(f'  ERROR: {e}')
            writer.writerow([rel.name, category_folder, f'ERROR: {e}', '', '', ''])
            out.flush()

print(f'\nDone! Dataset tokens successfully saved to {out_path}')


NameError: name 'Path' is not defined

In [ ]:
from google.colab import auth, drive
import os

auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)

# Points directly to the exact dataset folder you mentioned
DRIVE_IMAGE_FOLDER = 'fire_detection/DataSet'
DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_IMAGE_FOLDER}'

assert os.path.isdir(DRIVE_DIR), (
    f'Folder not found: {DRIVE_DIR}\n'
    f'Make sure "{DRIVE_IMAGE_FOLDER}" exists inside My Drive on Google Drive.'
)

IMAGE_DIR = f'{DRIVE_DIR}/Looks_Like_Fire'
print(f'Reading images directly from {IMAGE_DIR}')

total = sum(1 for _, _, fs in os.walk(IMAGE_DIR) for f in fs)
print(f'Found {total} file(s) in Looks_Like_Fire')


Mounted at /content/drive
Reading images directly from /content/drive/MyDrive/fire_detection/DataSet/Looks_Like_Fire
Found 100 file(s) in Looks_Like_Fire


In [ ]:
!pip install -q "transformers==4.45.2" accelerate einops timm Pillow opencv-python-headless numpy torch torchvision
print('Done.')


Done.


In [ ]:
import pathlib, sys

CACHE_ROOT = pathlib.Path.home() / '.cache/huggingface/modules/transformers_modules/OpenGVLab'

BAD_LINE  = 'dpr = [x.item() for x in torch.linspace(0, config.drop_path_rate, config.num_hidden_layers)]'
GOOD_LINE = ('_n   = int(config.num_hidden_layers)\n'
             '        _end = config.drop_path_rate\n'
             '        _end = _end if isinstance(_end, float) else 0.1\n'
             '        dpr  = [_end * i / max(_n - 1, 1) for i in range(_n)]')

patched = 0
for vit_file in CACHE_ROOT.rglob('modeling_intern_vit.py'):
    code = vit_file.read_text(encoding='utf-8')
    if BAD_LINE in code:
        vit_file.write_text(code.replace(BAD_LINE, GOOD_LINE), encoding='utf-8')
        print(f'Patched file: {vit_file}')
        patched += 1
    elif '_end = config.drop_path_rate' in code:
        print(f'Already patched: {vit_file}')
        patched += 1

evicted = [k for k in list(sys.modules) if 'internvl' in k.lower() or 'intern_vit' in k.lower()]
for k in evicted:
    del sys.modules[k]
print('Ready — run Cell 4 now.')


Patched file: /root/.cache/huggingface/modules/transformers_modules/OpenGVLab/InternVL2_5-4B/2cf4a8158bbc40d35015e7c63b527890de4d27b3/modeling_intern_vit.py
Ready — run Cell 4 now.


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_ID = 'OpenGVLab/InternVL2_5-4B'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE    = torch.bfloat16 if DEVICE == 'cuda' else torch.float32

print(f'Loading {MODEL_ID} on {DEVICE.upper()} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
model     = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=False,
).eval().to(DEVICE)

print(f'Model ready  |  device: {DEVICE.upper()}  |  dtype: {DTYPE}')


Loading OpenGVLab/InternVL2_5-4B on CUDA ...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model ready  |  device: CUDA  |  dtype: torch.bfloat16


In [ ]:
MAX_NEW_TOKENS   = 40
IMAGE_SIZE       = 448
MAX_TILES        = 6

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.wmv', '.m4v'}

# The highly-constrained 3-tier classification prompt under 10 words
PROMPT = (
    "Write a concise, 10-word description of this scene focusing on sources of light and atmospheric conditions. "
    "Clearly state exactly what is creating the light or haze (e.g., 'sun setting', 'active flames', 'dense smoke', "
    "or 'bright streetlamps'). Be highly specific about the environment."
)



# Common English stop words. 'no' and 'not' are intentionally removed so 'no_fire' isn't ruined!
STOPWORDS = {
    'a','an','the','is','are','was','were','be','been','being',
    'in','on','at','to','of','for','with','by','from','into',
    'and','or','but','as','its','it','this','that',
    'there','their','they','has','have','had','do','does','did',
    'i','we','you','he','she','over','under','near','between',
    'through','around','against','during','without','within',
}


In [ ]:
import time
from pathlib import Path
import cv2
import numpy as np
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(size=IMAGE_SIZE):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB')),
        T.Resize((size, size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def dynamic_preprocess(image, min_num=1, max_num=MAX_TILES, image_size=IMAGE_SIZE):
    w, h    = image.size
    aspect  = w / h
    ratios  = sorted(
        {(i, j) for n in range(min_num, max_num + 1)
                for i in range(1, n + 1)
                for j in range(1, n + 1)
                if min_num <= i * j <= max_num},
        key=lambda x: x[0] * x[1]
    )
    best = min(ratios, key=lambda r: abs(aspect - r[0] / r[1]))
    tw, th  = image_size * best[0], image_size * best[1]
    resized = image.resize((tw, th))
    tiles   = [
        resized.crop((
            (idx % best[0]) * image_size, (idx // best[0]) * image_size,
            ((idx % best[0]) + 1) * image_size, ((idx // best[0]) + 1) * image_size,
        ))
        for idx in range(best[0] * best[1])
    ]
    tiles.append(image.resize((image_size, image_size)))
    return tiles

def load_pixel_values(image):
    transform = build_transform()
    tiles     = dynamic_preprocess(image.convert('RGB'))
    return torch.stack([transform(t) for t in tiles]).to(DTYPE).to(DEVICE)

def get_phrase(image):
    pixel_values = load_pixel_values(image)
    gen_cfg      = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    response     = model.chat(tokenizer, pixel_values, PROMPT, gen_cfg)
    return response.strip()

def analyse_phrase(phrase):
    words             = phrase.split()
    meaningful_tokens = [w.strip('.,;:!?\'"').lower()
                         for w in words
                         if w.strip('.,;:!?\'"').lower() not in STOPWORDS
                         and w.strip('.,;:!?\'"')]
    return {
        'word_count'        : len(words),
        'meaningful_tokens' : meaningful_tokens,
        'token_count'       : len(meaningful_tokens),
    }

print('Helper functions ready.')


Helper functions ready.


In [ ]:
import csv
root     = Path(IMAGE_DIR)
# Export to CSV inside your fire_detection folder
out_path = Path('/content/drive/MyDrive/fire_detection/internvl_Looks_Like_Fire_tokens.csv')

all_files = sorted(
    p for p in root.rglob('*')
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS | VIDEO_EXTS
)

print(f'Processing {len(all_files)} file(s)...\n' + '─' * 60)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, 'w', encoding='utf-8', newline='') as out:
    writer = csv.writer(out)
    writer.writerow(['filename', 'category_folder', 'phrase', 'meaningful_tokens', 'word_count', 'meaningful_token_count'])

    for idx, path in enumerate(all_files, 1):
        rel = path.relative_to(root)
        # Pulls 'Safe_fire', 'Dangerous_fire', etc
        category_folder = rel.parent.name if rel.parent != Path('.') else 'Looks_Like_Fire'
        ext = path.suffix.lower()
        print(f'[{idx}/{len(all_files)}] {rel}')

        try:
            phrase = get_phrase(Image.open(path))
            info   = analyse_phrase(phrase)

            tokens_str = ' '.join(info['meaningful_tokens'])

            writer.writerow([
                rel.name,
                category_folder,
                phrase,
                tokens_str,
                info['word_count'],
                info['token_count']
            ])
            out.flush()

            print(f'  phrase      : {phrase!r}')
            print(f'  meaningful  : {info["meaningful_tokens"]}')

        except Exception as e:
            print(f'  ERROR: {e}')
            writer.writerow([rel.name, category_folder, f'ERROR: {e}', '', '', ''])
            out.flush()

print(f'\nDone! Dataset tokens successfully saved to {out_path}')


Processing 100 file(s)...
────────────────────────────────────────────────────────────
[1/100] 496.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


  phrase      : 'Sunset illuminates modern building, vibrant clouds, calm water reflecting hues.'
  meaningful  : ['sunset', 'illuminates', 'modern', 'building', 'vibrant', 'clouds', 'calm', 'water', 'reflecting', 'hues']
[2/100] 59.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


  phrase      : 'Bright streetlamps and dense smoke create a hazy atmosphere.'
  meaningful  : ['bright', 'streetlamps', 'dense', 'smoke', 'create', 'hazy', 'atmosphere']
[3/100] 61.jpg


...

  phrase      : 'Sun setting, casting warm hues; city lights and clouds create atmospheric glow.'
  meaningful  : ['sun', 'setting', 'casting', 'warm', 'hues', 'city', 'lights', 'clouds', 'create', 'atmospheric', 'glow']
[99/100] WEB10118.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


  phrase      : 'Sun setting over ocean, casting warm golden light and creating lens flare.'
  meaningful  : ['sun', 'setting', 'ocean', 'casting', 'warm', 'golden', 'light', 'creating', 'lens', 'flare']
[100/100] WEB10120.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


  phrase      : 'Sun setting, creating lens flare and soft light, with scattered clouds in the sky.'
  meaningful  : ['sun', 'setting', 'creating', 'lens', 'flare', 'soft', 'light', 'scattered', 'clouds', 'sky']

Done! Dataset tokens successfully saved to /content/drive/MyDrive/fire_detection/internvl_Looks_Like_Fire_tokens.csv
